In [0]:
# Discover your Unity Catalog structure
# Run this before anything else to find the correct volume path

# List all catalogs you have access to
print("=== CATALOGS ===")
display(spark.sql("SHOW CATALOGS"))

In [0]:
# Once you know your catalog name, run this (replace 'your_catalog' with the result above)
print("=== SCHEMAS ===")
display(spark.sql("SHOW SCHEMAS IN movie_recsys"))

In [0]:
# Once you know catalog + schema, run this to find the volume
print("=== VOLUMES ===")
display(spark.sql("SHOW VOLUMES IN movie_recsys.information_schema"))

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 01 — EDA & Filtering
# MAGIC ## CohortNova · Movie Recommendation System
# MAGIC
# MAGIC **Purpose:** Exploratory analysis of the Amazon Reviews 2023 (Movies & TV) dataset.
# MAGIC Produces the filtered, validated datasets consumed by Jobs 1–4.
# MAGIC
# MAGIC **Inputs:**
# MAGIC - `/Volumes/movie_recsys/data/raw/Movies_and_TV.jsonl.gz` — 17.3M reviews
# MAGIC - `/Volumes/movie_recsys/data/raw/meta_Movies_and_TV.jsonl.gz` — 748K items
# MAGIC
# MAGIC **Outputs (written to `/Volumes/movie_recsys/data/processed/`):**
# MAGIC - `reviews_5core.parquet` — reviews after 5-core filter
# MAGIC - `meta_clean.parquet` — metadata after deduplication & field normalisation
# MAGIC - `validation_report.json` — machine-readable counts for downstream assertions
# MAGIC
# MAGIC **Tags:** ACTUAL (Amazon Reviews 2023, McAuley Lab UCSD)
# MAGIC
# MAGIC **Run order:** This notebook must complete before Job 1 (embeddings) starts.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0 · Setup

# COMMAND ----------

import json
import re
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ── paths ────────────────────────────────────────────────────────────────────
RAW_DIR        = "/Volumes/movie_recsys/data/raw"
PROCESSED_DIR  = "/Volumes/movie_recsys/data/outputs"
REVIEWS_RAW    = f"{RAW_DIR}/Movies_and_TV.jsonl.gz"
META_RAW       = f"{RAW_DIR}/meta_Movies_and_TV.jsonl.gz"
REVIEWS_OUT    = f"{PROCESSED_DIR}/reviews_5core.parquet"
META_OUT       = f"{PROCESSED_DIR}/meta_clean.parquet"
REPORT_OUT     = f"{PROCESSED_DIR}/validation_report.json"

# ── config ───────────────────────────────────────────────────────────────────
# CONFIG PARAM — 5-core threshold. Change here without redeployment.
CORE_THRESHOLD = 5          # minimum interactions for user AND item to be retained
RATING_MIN     = 1.0
RATING_MAX     = 5.0
SAMPLE_SEED    = 42

dbutils.fs.mkdirs(PROCESSED_DIR)
print(f"Processed dir: {PROCESSED_DIR}")
print(f"5-core threshold: {CORE_THRESHOLD}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1 · Load Raw Reviews

# COMMAND ----------

# Amazon 2023 JSONL schema (reviews)
review_schema = T.StructType([
    T.StructField("rating",      T.FloatType(),  True),
    T.StructField("title",       T.StringType(), True),
    T.StructField("text",        T.StringType(), True),
    T.StructField("images",      T.ArrayType(T.StringType()), True),
    T.StructField("asin",        T.StringType(), True),   # item id
    T.StructField("parent_asin", T.StringType(), True),
    T.StructField("user_id",     T.StringType(), True),
    T.StructField("timestamp",   T.LongType(),   True),   # milliseconds since epoch
    T.StructField("helpful_vote",T.LongType(),   True),
    T.StructField("verified_purchase", T.BooleanType(), True),
])

reviews_raw = (
    spark.read
         .schema(review_schema)
         .json(REVIEWS_RAW)
)

raw_count = reviews_raw.count()
print(f"Raw review count: {raw_count:,}")


In [0]:
display(reviews_raw.limit(5))

In [0]:
reviews_raw = reviews_raw.drop("asin")

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 2 · Timestamp Parsing & Validation
# MAGIC
# MAGIC Amazon 2023 timestamps are **milliseconds** since Unix epoch (int64).
# MAGIC We convert to UTC datetime and extract date parts for EDA.
# MAGIC Rows with null or out-of-range timestamps are flagged and dropped.

# COMMAND ----------

# Convert ms → UTC datetime
reviews = (
    reviews_raw
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("parent_asin").isNotNull())
    .filter(F.col("rating").between(RATING_MIN, RATING_MAX))
    .withColumn(
        "timestamp_ms",
        F.col("timestamp").cast(T.LongType())
    )
    .withColumn(
        "event_ts",
        (F.col("timestamp_ms") / 1000).cast(T.TimestampType())
    )
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_year",  F.year("event_ts"))
    .withColumn("event_month", F.month("event_ts"))
)

# Sanity-check timestamp range
ts_stats = reviews.select(
    F.min("event_ts").alias("earliest"),
    F.max("event_ts").alias("latest"),
    F.sum(F.col("event_ts").isNull().cast("int")).alias("null_ts_count"),
).collect()[0]

print(f"Timestamp range : {ts_stats['earliest']}  →  {ts_stats['latest']}")
print(f"Null timestamps : {ts_stats['null_ts_count']:,}")

# Drop rows where timestamp conversion failed (edge-case malformed rows)
null_ts = ts_stats["null_ts_count"]
reviews = reviews.filter(F.col("event_ts").isNotNull())
print(f"Dropped {null_ts:,} null-timestamp rows. Remaining: {reviews.count():,}")

# Timestamp range : 1997-08-24 02:46:15  →  2023-09-12 22:13:12.711000
# Null timestamps : 0
# Dropped 0 null-timestamp rows. Remaining: 17,328,314

In [0]:
display(reviews.limit(5))

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ### 2a · Review volume over time
# MAGIC Expected: steady growth from ~2000, acceleration post-2015, tail into 2023.

# COMMAND ----------

yearly = (
    reviews
    .groupBy("event_year")
    .count()
    .orderBy("event_year")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(yearly["event_year"], yearly["count"] / 1e6, color="#1D9E75", width=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Reviews (millions)")
ax.set_title("Amazon Movies & TV — review volume by year", fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1fM"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3 · Rating Distribution

# COMMAND ----------

rating_dist = (
    reviews
    .groupBy("rating")
    .count()
    .orderBy("rating")
    .toPandas()
)

total = rating_dist["count"].sum()
rating_dist["pct"] = (rating_dist["count"] / total * 100).round(1)

print("Rating distribution:")
print(rating_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(
    rating_dist["rating"].astype(str),
    rating_dist["pct"],
    color="#3B8BD4", width=0.6
)
for bar, pct in zip(bars, rating_dist["pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4, f"{pct}%",
            ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Rating")
ax.set_ylabel("% of reviews")
ax.set_title("Rating distribution", fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [0]:
# reviews.columns

In [0]:
display(reviews.count())

In [0]:
# from pyspark.sql.functions import col
# from pyspark.sql import functions as F

# compare asin and parent_asin: asin is different derative of parent_asin, say, wide screen vs 4k with blueray

# # 1. Define the comparison
# # eqNullSafe handles nulls correctly (null <=> null is True)
# df_compared = reviews.withColumn("is_identical", F.col("asin") == F.col("parent_asin")) \
#                 .withColumn("is_identical_nullsafe", F.col("asin").eqNullSafe(F.col("parent_asin")))

# # 2. Get the counts
# stats = reviews.select(
#     F.sum(F.when(F.col("asin").eqNullSafe(F.col("parent_asin")), 1).otherwise(0)).alias("identical_count"),
#     F.sum(F.when(~F.col("asin").eqNullSafe(F.col("parent_asin")), 1).otherwise(0)).alias("different_count")
# ).collect()[0]

# identical = stats["identical_count"]
# different = stats["different_count"]

# # 3. Final Judgement
# if different == 0:
#     print(f"Columns are perfectly identical. Total rows: {identical}")
# else:
#     print(f"Columns are NOT identical.")
#     print(f"Identical rows: {identical:,}")
#     print(f"Different rows: {different:,}")

# # Columns are NOT identical.
# # Identical rows: 17,326,992
# # Different rows: 1,322

In [0]:
# NOTE FOR PRODUCT MEMO:
# Strong J-curve (5-star dominance) is normal for Amazon. The CF model will
# handle this via SVD's implicit centering on mean ratings per user.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4 · 5-Core Filter
# MAGIC
# MAGIC **Definition:** retain only users with ≥ 5 reviews AND items with ≥ 5 reviews.
# MAGIC Applied iteratively until convergence — removing sparse users exposes sparse
# MAGIC items and vice versa. Typically converges in 2–3 passes on this dataset.
# MAGIC
# MAGIC **Rationale:** SVD performs poorly on very sparse users. The 5-core removes
# MAGIC noise while preserving meaningful collaborative signals.

# COMMAND ----------

def apply_5core(df, threshold=CORE_THRESHOLD, max_iter=10):
    """
    Iterative 5-core filter -- Databricks Serverless-compatible.
    Returns (filtered_df, iteration_log).

    Serverless restrictions that ruled out earlier approaches:
      - cache() / persist() : PERSIST TABLE not supported (0A000)
      - localCheckpoint()   : requires spark.sparkContext (JVM_ATTRIBUTE_NOT_SUPPORTED)

    Solution: each iteration writes to a Delta scratch table and reads it back,
    truncating the lineage DAG using only the public DataFrame API.

    Two scratch paths are used:
      _5core_scratch : overwritten every pass; holds the working set.
      _5core_final   : written once at convergence; what the caller receives.

    Both are deleted AFTER all calling-side .count()/.distinct() calls complete.
    Deleting inside this function would invalidate the returned DataFrame because
    Delta scans are lazy -- the files must still exist when the caller reads them.
    """
    SCRATCH_PATH = f"{PROCESSED_DIR}/_5core_scratch"
    FINAL_PATH   = f"{PROCESSED_DIR}/_5core_final"

    iteration_log = []
    current = df

    for i in range(1, max_iter + 1):
        n_before = current.count()

        user_counts = (
            current.groupBy("user_id")
                   .count()
                   .filter(F.col("count") >= threshold)
                   .select("user_id")
        )
        item_counts = (
            current.groupBy("parent_asin")
                   .count()
                   .filter(F.col("count") >= threshold)
                   .select("parent_asin")
        )

        filtered = (
            current
            .join(user_counts, on="user_id", how="inner")
            .join(item_counts, on="parent_asin",    how="inner")
        )

        # Overwrite scratch each pass -- breaks the lineage DAG.
        (
            filtered
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(SCRATCH_PATH)
        )

        current = spark.read.format("delta").load(SCRATCH_PATH)

        n_after = current.count()
        dropped = n_before - n_after

        iteration_log.append({
            "iteration":    i,
            "rows_before":  n_before,
            "rows_after":   n_after,
            "rows_dropped": dropped,
        })
        print(f"  Pass {i}: {n_before:,} -> {n_after:,}  (dropped {dropped:,})")

        if dropped == 0:
            print(f"  Converged after {i} pass(es).")
            break

    # Write converged result to a stable final path.
    # The caller reads from here; scratch can then be safely deleted.
    (
        current
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(FINAL_PATH)
    )
    final_df = spark.read.format("delta").load(FINAL_PATH)

    return final_df, iteration_log




In [0]:
print(f"Applying {CORE_THRESHOLD}-core filter...")
reviews_5core, core_log = apply_5core(reviews, threshold=CORE_THRESHOLD)

# All three actions scan _5core_final, which still exists at this point.
n_users_5core   = reviews_5core.select("user_id").distinct().count()
n_items_5core   = reviews_5core.select("parent_asin").distinct().count()
n_reviews_5core = reviews_5core.count()

print(f"\n5-core result:")
print(f"  Reviews : {n_reviews_5core:,}  ({n_reviews_5core/raw_count*100:.1f}% of raw)")
print(f"  Users   : {n_users_5core:,}")
print(f"  Items   : {n_items_5core:,}")
print(f"  Sparsity: {1 - n_reviews_5core / (n_users_5core * n_items_5core):.6f}")

In [0]:
display(reviews_5core.limit(5))

In [0]:
# 2. Get the counts
# display(reviews_5core.filter(F.col("asin").isNull()).count())
# 0
# reviews_5core = reviews_5core.withColumn("stz_asin", F.col("asin"))

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 5 · Load & Clean Metadata

# COMMAND ----------

# Amazon 2023 metadata schema (items)
meta_schema = T.StructType([
    T.StructField("main_category",   T.StringType(), True),
    T.StructField("title",           T.StringType(), True),
    T.StructField("average_rating",  T.FloatType(),  True),
    T.StructField("rating_number",   T.LongType(),   True),
    T.StructField("features",        T.ArrayType(T.StringType()), True),
    T.StructField("description",     T.ArrayType(T.StringType()), True),
    T.StructField("price",           T.StringType(), True),   # raw string — needs cleaning
    T.StructField("images",          T.ArrayType(T.StringType()), True),
    T.StructField("videos",          T.ArrayType(T.StringType()), True),
    T.StructField("store",           T.StringType(), True),
    T.StructField("categories",      T.ArrayType(T.StringType()), True),
    T.StructField("details",         T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("parent_asin",     T.StringType(), True),
    T.StructField("bought_together",  T.ArrayType(T.StringType()), True),
])

meta_raw = (
    spark.read
         .schema(meta_schema)
         .json(META_RAW)
)

meta_raw_count = meta_raw.count()
print(f"Raw metadata count: {meta_raw_count:,}")

In [0]:
display(meta_raw.limit(5))

In [0]:

# Sanity-check: confirm asin field exists and is populated
# asin_nulls = meta_raw.filter(F.col("asin").isNull()).count()
parent_asin_nulls = meta_raw.filter(F.col("parent_asin").isNull()).count()

# print(f"Rows with null asin (will be dropped): {asin_nulls:,}")
# print()

print(f"Rows with null asin (will be dropped): {parent_asin_nulls:,}")
print()


In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## 5 · Load & Clean Metadata
# MAGIC
# MAGIC Key finding from display(): the item identifier in Amazon 2023 metadata
# MAGIC is `parent_asin`, not `asin`. The `asin` column is null for all rows.
# MAGIC Reviews reference items via `asin` = meta's `parent_asin`.
# MAGIC Join key throughout: reviews.asin == meta.parent_asin.

# COMMAND ----------

# Print schema for debugging
print("Schema fields and types:")
for field in meta_raw.schema.fields:
    print(f"  {field.name:20s}: {field.dataType}")

# Confirm parent_asin is the populated identifier
null_parent = meta_raw.filter(F.col("parent_asin").isNull()).count()
print(f"Null parent_asin : {null_parent:,}   ← this should be near zero")
# print(f"Null asin        : {meta_raw.filter(F.col('asin').isNull()).count():,}   ← expected to be all rows")

# COMMAND ----------

@F.udf(T.FloatType())
def parse_price(raw):
    if not raw or raw.strip().lower() in ("none", ""):
        return None
    matches = re.findall(r"\$?([\d,]+\.?\d*)", raw.replace(",", ""))
    if not matches:
        return None
    try:
        return float(matches[0])
    except ValueError:
        return None

# Define schema for parsing image JSON strings
image_schema = T.StructType([
    T.StructField("thumb", T.StringType(), True),
    T.StructField("large", T.StringType(), True),
    T.StructField("variant", T.StringType(), True),
    T.StructField("hi_res", T.StringType(), True),
])

meta = (
    meta_raw
    .filter(F.col("parent_asin").isNotNull())
    .dropDuplicates(["parent_asin"])
    .withColumnRenamed("parent_asin", "item_id")
    .withColumn(
        "details_str",
        F.when(
            F.col("details").isNotNull(),
            F.to_json(F.col("details"))
        ).otherwise(F.lit(""))
    )
    .drop("details")
    .withColumn("price_raw",   F.col("price").cast(T.StringType()))
    .withColumn("price_float", parse_price(F.col("price").cast(T.StringType())))
    .withColumn("has_price",   F.col("price_float").isNotNull())
    .withColumn(
        "description_str",
        F.when(F.col("description").isNotNull(),
               F.concat_ws(" ", F.col("description"))
        ).otherwise(F.lit(""))
    )
    .withColumn(
        "primary_genre",
        F.when(F.coalesce(F.size(F.col("categories")), F.lit(0)) > 0,
               F.col("categories")[0]
        ).otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "genres_str",
        F.when(F.col("categories").isNotNull(),
               F.concat_ws(", ", F.col("categories"))
        ).otherwise(F.lit(""))
    )
    .withColumn(
        "poster_url",
        F.when(
            (F.col("images").isNotNull()) &
            (F.coalesce(F.size(F.col("images")), F.lit(0)) > 0),
            F.from_json(F.col("images")[0], image_schema)["large"]
        ).otherwise(F.lit(None).cast(T.StringType()))
    )
)

meta_count = meta.count()
print(f"meta row count after cleaning: {meta_count:,}")   # expect ~748K

In [0]:
display(meta.limit(5))

In [0]:

# COMMAND ----------
 
# Price coverage stats
price_stats = meta.select(
    F.count("*").alias("total"),
    F.sum(F.col("has_price").cast("int")).alias("has_price"),
    F.mean("price_float").alias("mean_price"),
    F.percentile_approx("price_float", 0.25).alias("p25"),
    F.percentile_approx("price_float", 0.50).alias("p50"),
    F.percentile_approx("price_float", 0.75).alias("p75"),
    F.max("price_float").alias("max_price"),
).collect()[0]
 
price_coverage = price_stats["has_price"] / price_stats["total"]
print(f"Price field coverage: {price_coverage:.1%}  ({price_stats['has_price']:,} / {price_stats['total']:,})")
 
# Guard: only print price stats if there are items with price
if price_stats["mean_price"] is not None:
    print(f"Price distribution (items with price):")
    print(f"  Mean : ${price_stats['mean_price']:.2f}")
    print(f"  P25  : ${price_stats['p25']:.2f}")
    print(f"  P50  : ${price_stats['p50']:.2f}")
    print(f"  P75  : ${price_stats['p75']:.2f}")
    print(f"  Max  : ${price_stats['max_price']:.2f}")
else:
    print("  No items with parseable price found.")
 
# Blueprint decision gate
if price_coverage >= 0.20:
    print(f"\n✓ Price coverage {price_coverage:.1%} >= 20% threshold.")
    print("  Tag: ACTUAL — use Amazon price data for revenue tier estimation.")
    price_tag = "ACTUAL"
else:
    print(f"\n⚠ Price coverage {price_coverage:.1%} < 20% threshold.")
    print("  Tag: SIMULATED — fall back to industry benchmark revenue tiers.")
    price_tag = "SIMULATED"
 

In [0]:

total     = price_stats["total"]       # guaranteed non-null (count never null)
has_price = price_stats["has_price"] or 0  # coerce None → 0 as final safety net

def fmt(val):
    return f"${val:.2f}" if val is not None else "N/A"

price_coverage = has_price / total
print(f"Price field coverage: {price_coverage:.1%}  ({has_price:,} / {total:,})")
print(f"Price distribution (items with price):")
print(f"  Mean : {fmt(price_stats['mean_price'])}")
print(f"  P25  : {fmt(price_stats['p25'])}")
print(f"  P50  : {fmt(price_stats['p50'])}")
print(f"  P75  : {fmt(price_stats['p75'])}")
print(f"  Max  : {fmt(price_stats['max_price'])}")

# Blueprint decision gate — tag carried forward into validation_report.json
if price_coverage >= 0.20:
    print(f"\n✓ Price coverage {price_coverage:.1%} >= 20% threshold.")
    print("  Tag: ACTUAL — use Amazon price data for revenue tier estimation.")
    price_tag = "ACTUAL"
else:
    print(f"\n⚠ Price coverage {price_coverage:.1%} < 20% threshold.")
    print("  Tag: SIMULATED — fall back to industry benchmark revenue tiers.")
    price_tag = "SIMULATED"

# Persist coverage values for validation report (Section 10)
price_coverage_pct  = round(price_coverage * 100, 1)
price_mean          = float(price_stats["mean_price"]) if price_stats["mean_price"] is not None else None
price_p50           = float(price_stats["p50"])        if price_stats["p50"]        is not None else None


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ### 5b · Revenue Tier Assignment
# MAGIC
# MAGIC Based on Section 2.3.2 of the blueprint:
# MAGIC - Free-with-ads: $0.00
# MAGIC - Rental: ~$3.99
# MAGIC - Purchase: ~$14.99

# COMMAND ----------

# Assign revenue tier based on parsed price
meta = meta.withColumn(
    "revenue_tier",
    F.when(F.col("price_float").isNull(), "unknown")
     .when(F.col("price_float") == 0.0,  "free_with_ads")
     .when(F.col("price_float") <= 5.99, "rental")
     .when(F.col("price_float") <= 25.0, "purchase")
     .otherwise("premium")
)

tier_dist = (
    meta
    .groupBy("revenue_tier")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

total_meta = tier_dist["count"].sum()
tier_dist["pct"] = (tier_dist["count"] / total_meta * 100).round(1)
print("Revenue tier distribution:")
print(tier_dist.to_string(index=False))


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 6 · Genre Distribution
# MAGIC
# MAGIC `categories` field is a list of tags from Amazon's browse taxonomy.
# MAGIC We use the first element as the primary genre for CB embeddings and EDA.

# COMMAND ----------

genre_dist = (
    meta
    .groupBy("primary_genre")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(
    genre_dist["primary_genre"][::-1],
    genre_dist["count"][::-1] / 1000,
    color="#534AB7", height=0.65
)
ax.set_xlabel("Items (thousands)")
ax.set_title("Top 20 genres by item count (metadata)", fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0fK"))
plt.tight_layout()
plt.show()

# How many distinct genres?
n_genres = meta.select("primary_genre").distinct().count()
print(f"Distinct primary genres: {n_genres:,}")


In [0]:
# meta.columns

In [0]:
meta = meta.withColumn("parent_asin", F.col("item_id"))

In [0]:

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6a · Genre distribution in 5-core reviews
# MAGIC
# MAGIC Cross-check: are the same genres represented in the filtered review set?
# MAGIC A mismatch would indicate the 5-core filter skews toward certain content types.

# COMMAND ----------

reviews_with_genre = (
    reviews_5core
    .join(
        meta.select("parent_asin", "primary_genre"),
        on="parent_asin", how="left"
    )
    .withColumn(
        "primary_genre",
        F.coalesce(F.col("primary_genre"), F.lit("Unknown"))
    )
)

genre_reviews = (
    reviews_with_genre
    .groupBy("primary_genre")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(genre_dist["primary_genre"][:15][::-1],
             genre_dist["count"][:15][::-1] / 1000,
             color="#534AB7", height=0.65)
axes[0].set_title("Items by genre (all metadata)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Items (K)")
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].barh(genre_reviews["primary_genre"][::-1],
             genre_reviews["count"][::-1] / 1000,
             color="#1D9E75", height=0.65)
axes[1].set_title("Reviews by genre (5-core)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Reviews (K)")
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("Genre distribution: all metadata vs 5-core reviews", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


In [0]:

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7 · Metadata Coverage for CB Embeddings
# MAGIC
# MAGIC The CB engine embeds: `title + genres_str + description_str + most_helpful_review`.
# MAGIC Check field availability across the full metadata set.

# COMMAND ----------

# Most helpful review per item (used in CB embedding input)
most_helpful = (
    reviews_5core
    .orderBy(F.desc("helpful_vote"))
    .groupBy("parent_asin")
    .agg(F.first("text").alias("most_helpful_review"))
)

meta_with_review = meta.join(most_helpful, on="parent_asin", how="left")

coverage = meta_with_review.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("title").isNotNull()            & (F.col("title")            != ""), 1).otherwise(0)).alias("has_title"),
    F.sum(F.when(F.col("genres_str").isNotNull()       & (F.col("genres_str")       != ""), 1).otherwise(0)).alias("has_genres"),
    F.sum(F.when(F.col("description_str").isNotNull()  & (F.col("description_str")  != ""), 1).otherwise(0)).alias("has_description"),
    F.sum(F.when(F.col("most_helpful_review").isNotNull() & (F.col("most_helpful_review") != ""), 1).otherwise(0)).alias("has_review"),
).collect()[0]

n = coverage["total"]
print("Embedding field coverage (all metadata items):")
print(f"  Title              : {coverage['has_title']/n:6.1%}  ({coverage['has_title']:,})")
print(f"  Genres             : {coverage['has_genres']/n:6.1%}  ({coverage['has_genres']:,})")
print(f"  Description        : {coverage['has_description']/n:6.1%}  ({coverage['has_description']:,})")
print(f"  Most helpful review: {coverage['has_review']/n:6.1%}  ({coverage['has_review']:,})")
print()
print("Items missing title or genres will be bridged via TMDB API in Job 1.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 7a · Embedding input text — spot check
# MAGIC
# MAGIC Construct the embedding input string for 5 sample items to confirm the
# MAGIC concatenation logic before passing 500K items to sentence-transformers.

# COMMAND ----------

def build_embedding_input(title, genres_str, description_str, review_text, max_review_tokens=256):
    """
    Construct the text fed to sentence-transformers/all-MiniLM-L6-v2.
    Mirrors the logic in src/features.py.
    review_text is truncated at ~max_review_tokens words (proxy for tokens).
    """
    parts = []
    if title:            parts.append(title.strip())
    if genres_str:       parts.append(genres_str.strip())
    if description_str:  parts.append(description_str.strip())
    if review_text:
        words = review_text.split()[:max_review_tokens]
        parts.append(" ".join(words))
    return " | ".join(p for p in parts if p)


samples = (
    meta_with_review
    .filter(F.col("title").isNotNull())
    .select("parent_asin", "title", "genres_str", "description_str", "most_helpful_review")
    .limit(5)
    .toPandas()
)

print("Embedding input spot-check (5 items):\n")
for _, row in samples.iterrows():
    emb_text = build_embedding_input(
        row["title"], row["genres_str"],
        row["description_str"], row["most_helpful_review"]
    )
    print(f"parent_asin  : {row['parent_asin']}")
    print(f"Title : {row['title']}")
    print(f"Input : {emb_text[:300]}{'...' if len(emb_text) > 300 else ''}")
    print(f"Length: {len(emb_text.split())} words")
    print("-" * 70)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8 · Cohort Nova Eligibility Check
# MAGIC
# MAGIC Cohort Nova requires users with 25+ lifetime ratings (Section SIMULATED DATASETS).
# MAGIC Confirm sample size is large enough to power the A/B test.
# MAGIC
# MAGIC Required sample per arm (from power calculation at MDE=2pp, α=0.05, power=0.80): ~4,800.
# MAGIC We need at least 9,600 eligible users total.

# COMMAND ----------

cohort_eligible = (
    reviews_5core
    .groupBy("user_id")
    .agg(F.count("*").alias("n_ratings"))
    .filter(F.col("n_ratings") >= 25)
)

n_eligible = cohort_eligible.count()
print(f"Cohort Nova eligible users (25+ ratings): {n_eligible:,}")

# Rough power check — required n per arm for MDE=2pp, alpha=0.05, power=0.80
# Formula: n ≈ 2 × (z_α/2 + z_β)² × p(1-p) / δ²
# z_0.025=1.96, z_0.20=0.84, p≈0.50 (worst case), δ=0.02
z_alpha = 1.96
z_beta  = 0.84
p_base  = 0.50
delta   = 0.02
n_required = int(2 * ((z_alpha + z_beta) ** 2) * p_base * (1 - p_base) / (delta ** 2))

print(f"Required users per arm (MDE=2pp): {n_required:,}")
print(f"Required total                  : {n_required*2:,}")
if n_eligible >= n_required * 2:
    print(f"✓ Sufficient sample for Cohort Nova simulation.")
else:
    print(f"⚠ Eligible pool ({n_eligible:,}) < required ({n_required*2:,}).")
    print(f"  Options: lower MDE, reduce core threshold, or note as limitation.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9 · Write Filtered Outputs

# COMMAND ----------

# Write 5-core reviews
(
    reviews_5core
    .select(
        "user_id", "parent_asin", "rating",
        "event_ts", "event_date", "event_year", "event_month",
        "helpful_vote", "verified_purchase", "text"
    )
    .write
    .mode("overwrite")
    .parquet(REVIEWS_OUT)
)
print(f"✓ reviews_5core.parquet written → {REVIEWS_OUT}")

# Write clean metadata
(
    meta
    .select(
        "parent_asin", "title", "primary_genre", "genres_str",
        "description_str", "price_raw", "price_float",
        "has_price", "revenue_tier", "average_rating", "rating_number"
    )
    .write
    .mode("overwrite")
    .parquet(META_OUT)
)
print(f"✓ meta_clean.parquet written → {META_OUT}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10 · Validation Report
# MAGIC
# MAGIC Machine-readable JSON consumed by downstream notebooks and the README.

# COMMAND ----------

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "raw": {
        "reviews": raw_count,
        "metadata": meta_raw_count,
    },
    "after_5core": {
        "reviews":  n_reviews_5core,
        "users":    n_users_5core,
        "items":    n_items_5core,
        "sparsity": round(1 - n_reviews_5core / (n_users_5core * n_items_5core), 6),
        "pct_of_raw_reviews": round(n_reviews_5core / raw_count * 100, 1),
    },
    "timestamp_parsing": {
        "null_ts_dropped": int(null_ts),
        "earliest": str(ts_stats["earliest"]),
        "latest":   str(ts_stats["latest"]),
    },
    "price_field": {
        "coverage_pct": round(price_coverage * 100, 1),
        "tag": price_tag,
        "mean": round(float(price_stats["mean_price"]), 2) if price_stats["mean_price"] else None,
        "p50":  round(float(price_stats["p50"]), 2) if price_stats["p50"] else None,
    },
    "genres": {
        "distinct_primary": n_genres,
    },
    "cohort_nova": {
        "eligible_users_25plus": n_eligible,
        "required_per_arm":      n_required,
        "sufficient":            n_eligible >= n_required * 2,
    },
    "embedding_coverage": {
        "total_items":    int(n),
        "has_title_pct":       round(coverage["has_title"]       / n * 100, 1),
        "has_genres_pct":      round(coverage["has_genres"]      / n * 100, 1),
        "has_description_pct": round(coverage["has_description"] / n * 100, 1),
        "has_review_pct":      round(coverage["has_review"]      / n * 100, 1),
    },
    "5core_log": core_log,
}

# Write to DBFS
report_json = json.dumps(report, indent=2, default=str)
dbutils.fs.put(REPORT_OUT, report_json, overwrite=True)
print(f"✓ validation_report.json written → {REPORT_OUT}")

# Pretty-print summary
print("\n" + "=" * 60)
print("VALIDATION REPORT SUMMARY")
print("=" * 60)
print(f"Raw reviews         : {report['raw']['reviews']:>12,}")
print(f"After 5-core        : {report['after_5core']['reviews']:>12,}  "
      f"({report['after_5core']['pct_of_raw_reviews']}% retained)")
print(f"Unique users        : {report['after_5core']['users']:>12,}")
print(f"Unique items        : {report['after_5core']['items']:>12,}")
print(f"Matrix sparsity     : {report['after_5core']['sparsity']:>12.6f}")
print(f"Price coverage      : {report['price_field']['coverage_pct']:>11}%  [{report['price_field']['tag']}]")
print(f"Primary genres      : {report['genres']['distinct_primary']:>12,}")
print(f"Cohort Nova eligible: {report['cohort_nova']['eligible_users_25plus']:>12,}  "
      f"({'✓ sufficient' if report['cohort_nova']['sufficient'] else '⚠ borderline'})")
print("=" * 60)
print("\nAll outputs ready. Proceed to Job 1 (embeddings).")

In [0]:
# most_helpful.count()
# 200152

In [0]:
display(most_helpful.limit(5))

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## 11 · Summary of Findings
# MAGIC
# MAGIC This section summarizes key findings from the EDA and filtering steps above.
# MAGIC The summary is saved to a text file for review and documentation.

# COMMAND ----------

summary_lines = [
    "Amazon Movies & TV — EDA & Filtering Summary",
    "============================================",
    "",
    f"Raw review count: {raw_count:,}",
    f"Raw metadata count: {meta_raw_count:,}",
    "",
    f"Timestamp range: {ts_stats['earliest']} → {ts_stats['latest']}",
    f"Null timestamps dropped: {null_ts:,}",
    "",
    f"5-core filter applied (threshold={CORE_THRESHOLD}):",
    f"  Reviews retained: {n_reviews_5core:,} ({report['after_5core']['pct_of_raw_reviews']}% of raw)",
    f"  Unique users: {n_users_5core:,}",
    f"  Unique items: {n_items_5core:,}",
    f"  Matrix sparsity: {report['after_5core']['sparsity']:.6f}",
    "",
    f"User activity (post 5-core):",
    f"  Users with 25+ ratings: {report['cohort_nova']['eligible_users_25plus']:,} "
    f"({'✓ sufficient' if report['cohort_nova']['sufficient'] else '⚠ borderline'})",
    "",
    f"Price field coverage: {report['price_field']['coverage_pct']}% [{report['price_field']['tag']}]",
    f"  Mean price: {report['price_field']['mean']}",
    f"  Median price (p50): {report['price_field']['p50']}",
    "",
    f"Distinct primary genres: {report['genres']['distinct_primary']:,}",
    "",
    "Embedding field coverage (metadata):",
    f"  Title: {report['embedding_coverage']['has_title_pct']}%",
    f"  Genres: {report['embedding_coverage']['has_genres_pct']}%",
    f"  Description: {report['embedding_coverage']['has_description_pct']}%",
    f"  Most helpful review: {report['embedding_coverage']['has_review_pct']}%",
    "",
    "All outputs written:",
    f"  reviews_5core.parquet → {REVIEWS_OUT}",
    f"  meta_clean.parquet → {META_OUT}",
    f"  validation_report.json → {REPORT_OUT}",
    "",
    "Proceed to Job 1 (embeddings).",
]

summary_text = "\n".join(summary_lines)
SUMMARY_PATH = f"{PROCESSED_DIR}/eda_summary.txt"
dbutils.fs.put(SUMMARY_PATH, summary_text, overwrite=True)
print(f"✓ Summary of findings written → {SUMMARY_PATH}")

In [0]:
## Evaluate works before Job1 

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 12 · Post-Filter Distribution Report (Excel-Ready Tables)
# MAGIC Each section displays as a table. Click any table, copy (Ctrl+C), and paste into Excel.

# COMMAND ----------

import pandas as pd
from pyspark.sql import functions as F


PROCESSED_DIR = "/Volumes/movie_recsys/data/outputs"
REVIEWS_OUT   = f"{PROCESSED_DIR}/reviews_5core.parquet"
META_OUT      = f"{PROCESSED_DIR}/meta_clean.parquet"


reviews = spark.read.parquet(REVIEWS_OUT)
meta    = spark.read.parquet(META_OUT)

# ── 1. REVIEWS: rating distribution ──────────────────────────
print("\n=== RATING DISTRIBUTION ===")
rating_dist = (
    reviews.groupBy("rating").count().orderBy("rating").toPandas()
)
total_reviews = rating_dist["count"].sum()
rating_dist["pct"] = (rating_dist["count"] / total_reviews * 100).round(1)
display(rating_dist)

# ── 2. REVIEWS: user activity buckets ────────────────────────
print("\n=== USER ACTIVITY (ratings per user) ===")
user_counts = reviews.groupBy("user_id").count().withColumnRenamed("count", "n_ratings")
total_users = user_counts.count()

user_buckets = []
for label, lo, hi in [("5-9",5,10),("10-24",10,25),("25-49",25,50),("50-99",50,100),("100+",100,99999)]:
    n = user_counts.filter((F.col("n_ratings") >= lo) & (F.col("n_ratings") < hi)).count()
    user_buckets.append({"bucket": label, "user_count": n, "pct": round(n / total_users * 100, 1)})
user_buckets.append({"bucket": "TOTAL", "user_count": total_users, "pct": 100.0})
user_activity_df = pd.DataFrame(user_buckets)
display(user_activity_df)

# ── 3. REVIEWS: item popularity buckets ──────────────────────
print("\n=== ITEM POPULARITY (ratings per item) ===")
item_counts = reviews.groupBy("parent_asin").count().withColumnRenamed("count", "n_ratings")
total_items = item_counts.count()

item_buckets = []
for label, lo, hi in [("5-9",5,10),("10-24",10,25),("25-99",25,100),("100-499",100,500),("500+",500,99999)]:
    n = item_counts.filter((F.col("n_ratings") >= lo) & (F.col("n_ratings") < hi)).count()
    item_buckets.append({"bucket": label, "item_count": n, "pct": round(n / total_items * 100, 1)})
item_buckets.append({"bucket": "TOTAL", "item_count": total_items, "pct": 100.0})
item_popularity_df = pd.DataFrame(item_buckets)
display(item_popularity_df)

# ── 4. META: price percentiles ───────────────────────────────
print("\n=== PRICE PERCENTILES (items with price) ===")
p = (
    meta.filter(F.col("has_price") == True)
    .select(
        F.count("*").alias("n"),
        F.min("price_float").alias("min"),
        F.expr("percentile(price_float, 0.10)").alias("p10"),
        F.expr("percentile(price_float, 0.25)").alias("p25"),
        F.expr("percentile(price_float, 0.50)").alias("p50"),
        F.mean("price_float").alias("mean"),
        F.expr("percentile(price_float, 0.75)").alias("p75"),
        F.expr("percentile(price_float, 0.90)").alias("p90"),
        F.expr("percentile(price_float, 0.99)").alias("p99"),
        F.max("price_float").alias("max"),
        F.stddev("price_float").alias("std"),
    ).collect()[0]
)

price_stats = []
for stat in ["n","min","p10","p25","p50","mean","p75","p90","p99","max","std"]:
    val = p[stat]
    price_stats.append({"statistic": stat, "value_usd": round(float(val), 2) if val is not None else "N/A"})
price_percentiles_df = pd.DataFrame(price_stats)
display(price_percentiles_df)

print("\n=== PRICE BUCKETS ===")
price_buckets = []
for label, lo, hi in [("<$5",0,5),("$5-9.99",5,10),("$10-19.99",10,20),("$20-49.99",20,50),("$50-99.99",50,100),("$100+",100,99999)]:
    n = meta.filter(F.col("has_price") & (F.col("price_float") >= lo) & (F.col("price_float") < hi)).count()
    price_buckets.append({"bucket": label, "item_count": n, "pct": round(n / p["n"] * 100, 1)})
price_buckets_df = pd.DataFrame(price_buckets)
display(price_buckets_df)

# ── 5. META: genre distribution ───────────────────────────────
print("\n=== TOP 20 PRIMARY GENRES ===")
genre_dist = (
    meta.filter(F.col("primary_genre").isNotNull())
    .groupBy("primary_genre").count()
    .orderBy(F.desc("count")).limit(20).toPandas()
)
genre_dist["pct_of_all_items"] = (genre_dist["count"] / total_items * 100).round(1)
genre_dist.columns = ["genre", "item_count", "pct_of_all_items"]
display(genre_dist)

# ── 6. META: item average rating distribution ─────────────────
print("\n=== ITEM AVERAGE RATING DISTRIBUTION ===")
avg_dist = (
    meta.filter(F.col("average_rating").isNotNull())
    .select(F.round("average_rating", 1).alias("avg_rating"))
    .groupBy("avg_rating").count().orderBy("avg_rating").toPandas()
)
total_rated = avg_dist["count"].sum()
avg_dist["pct"] = (avg_dist["count"] / total_rated * 100).round(1)
avg_dist.columns = ["avg_rating", "item_count", "pct"]
display(avg_dist)

# ── 7. REVIEWS: temporal distribution ────────────────────────
print("\n=== REVIEW VOLUME BY YEAR ===")
year_dist = (
    reviews.groupBy("event_year").count().orderBy("event_year").toPandas()
)
year_dist["pct"] = (year_dist["count"] / total_reviews * 100).round(1)
year_dist.columns = ["year", "review_count", "pct"]
display(year_dist)

print("\n✓ All tables displayed above. Click any table and copy (Ctrl+C) to paste into Excel.")

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 13 · Missing Rate Analysis (All Fields)
# MAGIC 
# MAGIC Complete inventory of missing/null values across reviews and metadata.

# COMMAND ----------

import pandas as pd
from pyspark.sql import functions as F

PROCESSED_DIR = "/Volumes/movie_recsys/data/outputs"
REVIEWS_OUT   = f"{PROCESSED_DIR}/reviews_5core.parquet"
META_OUT      = f"{PROCESSED_DIR}/meta_clean.parquet"

reviews = spark.read.parquet(REVIEWS_OUT)
meta    = spark.read.parquet(META_OUT)

# ── REVIEWS dataset missing rates ────────────────────────────
print("\n=== CALCULATING MISSING RATES ===")

reviews_total = reviews.count()
meta_total = meta.count()

missing_analysis = []

# Analyze REVIEWS fields
for col_name in reviews.columns:
    null_count = reviews.filter(F.col(col_name).isNull()).count()
    
    # For string fields, also check empty strings
    if dict(reviews.dtypes)[col_name] == 'string':
        empty_count = reviews.filter(
            (F.col(col_name).isNull()) | (F.col(col_name) == "")
        ).count()
        missing_count = empty_count
        missing_type = "null or empty"
    else:
        missing_count = null_count
        missing_type = "null"
    
    missing_rate = round(missing_count / reviews_total * 100, 2)
    
    missing_analysis.append({
        "dataset": "reviews",
        "field": col_name,
        "data_type": dict(reviews.dtypes)[col_name],
        "total_records": reviews_total,
        "missing_count": missing_count,
        "missing_rate_pct": missing_rate,
        "missing_type": missing_type
    })

# Analyze META fields
for col_name in meta.columns:
    null_count = meta.filter(F.col(col_name).isNull()).count()
    
    # For string fields, also check empty strings
    if dict(meta.dtypes)[col_name] == 'string':
        empty_count = meta.filter(
            (F.col(col_name).isNull()) | (F.col(col_name) == "")
        ).count()
        missing_count = empty_count
        missing_type = "null or empty"
    else:
        missing_count = null_count
        missing_type = "null"
    
    missing_rate = round(missing_count / meta_total * 100, 2)
    
    missing_analysis.append({
        "dataset": "meta",
        "field": col_name,
        "data_type": dict(meta.dtypes)[col_name],
        "total_records": meta_total,
        "missing_count": missing_count,
        "missing_rate_pct": missing_rate,
        "missing_type": missing_type
    })

# Create comprehensive DataFrame
missing_df = pd.DataFrame(missing_analysis)

# Sort by dataset, then by missing rate (descending)
missing_df = missing_df.sort_values(
    by=["dataset", "missing_rate_pct"], 
    ascending=[True, False]
).reset_index(drop=True)

print("\n=== MISSING RATE ANALYSIS (ALL FIELDS) ===")
print(f"Reviews dataset: {reviews_total:,} records")
print(f"Metadata dataset: {meta_total:,} records\n")
display(missing_df)

# ── Summary statistics ────────────────────────────────────────
print("\n=== SUMMARY BY DATASET ===")

summary_stats = missing_df.groupby('dataset').agg({
    'field': 'count',
    'missing_rate_pct': ['mean', 'median', 'max']
}).round(2)

summary_stats.columns = ['field_count', 'avg_missing_pct', 'median_missing_pct', 'max_missing_pct']
summary_stats = summary_stats.reset_index()
display(summary_stats)

# ── High missing rate fields (>50%) ──────────────────────────
print("\n=== HIGH MISSING RATE FIELDS (>50%) ===")
high_missing = missing_df[missing_df['missing_rate_pct'] > 50].copy()

if len(high_missing) > 0:
    display(high_missing[['dataset', 'field', 'missing_rate_pct', 'missing_count']])
else:
    print("✓ No fields with >50% missing rate")

# ── Complete fields (0% missing) ─────────────────────────────
print("\n=== COMPLETE FIELDS (0% missing) ===")
complete_fields = missing_df[missing_df['missing_rate_pct'] == 0].copy()

if len(complete_fields) > 0:
    display(complete_fields[['dataset', 'field', 'data_type']])
else:
    print("⚠ No fields are 100% complete")

print("\n✓ All tables can be copied to Excel. Use Ctrl+C after clicking any table.")